In [22]:
from mssql_python import connect
from psycopg2 import connect as pg_connect
import configparser
from pathlib import Path
import pandas as pd
import io

In [2]:
base_dir = Path.cwd().parent
conf_path = base_dir / 'pipeline.conf'

In [ ]:
# Obtendo parâmetros do azure sql

parser = configparser.ConfigParser()
parser.read(conf_path)
az_hostname = parser.get("azsql_config", "hostname")
az_username = parser.get("azsql_config", "username")
az_password = parser.get("azsql_config", "password")
az_dbname = parser.get("azsql_config", "database")

In [ ]:
connection_string = (
    f"Server={az_hostname};"
    f"Database={az_dbname};"
    f"UID={az_username};"
    f"PWD={az_password};"
    "Encrypt=yes;"
    "TrustServerCertificate=Yes;"   
)

In [16]:
try:
    with connect(connection_string) as conn:
        with conn.cursor() as cursor:

            cursor.execute("""
                SELECT 
                    c.*,
                    'dbo.nowrthwind' as origem
                FROM Customers c;
                """)
            rows= cursor.fetchall()
            columns = [column[0] for column in cursor.description]
            df = pd.DataFrame(rows, columns = columns)

except Exception as err:
    print(f"\nOcorreu um erro: {err}")

In [ ]:
# Obtendo parâmetros do pgsql

parser = configparser.ConfigParser()
parser.read(conf_path)
pg_hostname = parser.get("psql_config", "hostname")
pg_username = parser.get("psql_config", "username")
pg_password = parser.get("psql_config", "password")
pg_dbname = parser.get("psql_config", "database")
pg_port = parser.get("psql_config", "port")

In [21]:
pg_connection_url = f"postgresql://{pg_username}:{pg_password}@{pg_hostname}:{pg_port}/{pg_dbname}"

In [23]:
try:
    with pg_connect(pg_connection_url) as conn:
        with conn.cursor() as cursor:
            # Move os dados para o buffer
            csv_buffer = io.StringIO()
            df.write_csv(csv_buffer)
            csv_buffer.seek(0)

            # Executa o COPY para a tabela
            copy_sql = """
                COPY staging.customers (
                    CustomerID, CompanyName, ContactName, ContactTitle, 
                    Address, City, Region, PostalCode, Country, Phone, Fax, origem
                ) 
                FROM STDIN WITH CSV HEADER
            """
            cursor.copy_expert(copy_sql, csv_buffer)
            
except Exception as err:
    print(f"\nOcorreu um erro:{err}")


Ocorreu um erro:connection to server at "192.168.86.138", port 5431 failed: No route to host
	Is the server running on that host and accepting TCP/IP connections?

